In [ ]:
import pandas as pd
import json
from typing import Dict, Any, List
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [ ]:
df_ae = pd.read_csv("adae.csv")

In [ ]:
SCHEMA_DESCRIPTION = """
- AETERM: The reported term for the adverse event (e.g., Headache, Nausea).
- AESEV: The severity or intensity of the event (Values: MILD, MODERATE, SEVERE).
- AESOC: The System Organ Class or body system (e.g., Cardiac, Skin, Nervous system).
- USUBJID: Unique Subject Identifier.
"""

In [ ]:
class ClinicalTrialDataAgent:
    def __init__(self, api_key: str = None):
        self.parser = JsonOutputParser()
        
        if api_key:
            self.llm = ChatOpenAI(model="gpt-4o", openai_api_key=api_key, temperature=0)
        else:
            self.llm = None

    def get_structured_query(self, user_question: str) -> Dict[str, Any]:
        """Parses natural language into structured JSON."""
        
        prompt = ChatPromptTemplate.from_template(
            "You are a clinical data assistant. Map the user's question to the dataset schema.\n"
            "{schema}\n"
            "Question: {question}\n\n"
            "Return ONLY a JSON object with keys 'target_column' and 'filter_value'. "
            "Ensure 'filter_value' matches expected casing (e.g., MODERATE in uppercase)."
        )
        
        # Logic Flow: Prompt -> LLM -> Parse
        if self.llm:
            chain = prompt | self.llm | self.parser
            return chain.invoke({"schema": SCHEMA_DESCRIPTION, "question": user_question})
        else:
            return self._mock_llm_logic(user_question)

    def _mock_llm_logic(self, question: str) -> Dict[str, Any]:
        """Simple keyword routing to simulate LLM behavior."""
        q = question.lower()
        if "moderate" in q or "severity" in q:
            return {"target_column": "AESEV", "filter_value": "MODERATE"}
        elif "headache" in q:
            return {"target_column": "AETERM", "filter_value": "Headache"}
        elif "skin" in q or "body system" in q:
            return {"target_column": "AESOC", "filter_value": "Skin and subcutaneous tissue disorders"}
        return {"target_column": "UNKNOWN", "filter_value": "UNKNOWN"}

    def execute_query(self, df: pd.DataFrame, structured_query: Dict[str, Any]):
        """Applies the filter and returns unique subjects."""
        col = structured_query['target_column']
        val = structured_query['filter_value']
        
        if col not in df.columns:
            return 0, []

        mask = df[col].astype(str).str.contains(val, case=False, na=False)
        filtered_df = df[mask]
        
        unique_subjects = filtered_df['USUBJID'].unique().tolist()
        return len(unique_subjects), unique_subjects

In [ ]:
def run_tests():
    agent = ClinicalTrialDataAgent(api_key=None) 
    
    test_queries = [
        "Give me the subjects who had Adverse events of Moderate severity.",
        "Which patients reported a Headache?",
        "Identify subjects with events in the Skin."
    ]

    print(f"--- Clinical Assistant Test Results ---\n")
    for q in test_queries:
        structured_json = agent.get_structured_query(q)
        
        count, subjects = agent.execute_query(df_ae, structured_json)
        
        print(f"Query: {q}")
        print(f"Mapped To: {structured_json['target_column']} == {structured_json['filter_value']}")
        print(f"Result: {count} unique subject(s) found: {subjects}")
        print("-" * 40)

if __name__ == "__main__":
    run_tests()